In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
project_root = next(
    (path for path in (working_directory, *working_directory.parents)
     if (path / "pyproject.toml").is_file() and (path / "src").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Open this notebook from within the project directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# **CHRONOS-2 ETTh1 EXPERIMENT**

In [3]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from src.data.loader import load_dataset
from src.data.windows import make_last_window
from src.evaluation.metrics import evaluate_forecast, mase_per_series
from src.models.chronos import Chronos2Forecaster

In [6]:
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 96

### **Load Dataset**

In [5]:
dataset = load_dataset("ETTh1")
print(f"Dataset: {dataset.name}")
print(f"Observations: {dataset.n_observations}")
print(f"Variates: {dataset.n_variates}")
print(f"Frequency: {dataset.frequency}")

Dataset: ETTh1
Observations: 17420
Variates: 7
Frequency: h


### **Construct Holdout Window**

In [7]:
window = make_last_window(
    dataset=dataset,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH)
print(f"History shape: {window.history.shape}")
print(f"Target shape: {window.history.shape}")

History shape: (512, 7)
Target shape: (512, 7)


### **Load Model**

In [11]:
forecaster = Chronos2Forecaster()

Loading Chronos-2 from amazon/chronos-2 on cuda


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Chronos-2 loaded


### **Run Inference**

In [12]:
if torch.cuda.is_available():
    torch.cuda.synchronize()
start_time = time.perf_counter()

forecast = forecaster.predict(
    history=window.history,
    prediction_length=PREDICTION_LENGTH)

if torch.cuda.is_available():
    torch.cuda.synchronize()
inference_time = time.perf_counter() - start_time

In [13]:
print(f"Prediction shape: {forecast.predictions.shape}")

Prediction shape: (96, 7)


### **Validate Prediction**

In [14]:
if forecast.predictions.shape != window.target.shape:
    raise ValueError("Prediction shape does not match target shape.")
else:
    print("Prediction shape matches target shape.")

if not np.isfinite(forecast.predictions).all():
    raise ValueError("Forecast contains NaN or infinite values.")
else:
    print("Forecast does not contain NaN or infinite values.")

Prediction shape matches target shape.
Forecast does not contain NaN or infinite values.


### **Metrics**

In [15]:
if dataset.seasonal_period is None:
    raise ValueError("seasonal_period must be configured for MASE.")
else:
    print("seasonal_period has been configured for MASE.")

seasonal_period has been configured for MASE.


In [17]:
metrics = evaluate_forecast(
    y_true=window.target,
    y_pred=forecast.predictions,
    history=window.history,
    seasonal_period=dataset.seasonal_period)

In [18]:
per_series_mase = mase_per_series(
    y_true=window.target,
    y_pred=forecast.predictions,
    history=window.history,
    seasonal_period=dataset.seasonal_period)

In [19]:
metrics["inference_time_seconds"] = inference_time
metrics["context_length"] = CONTEXT_LENGTH
metrics["prediction_length"] = PREDICTION_LENGTH
metrics["dataset"] = dataset.name
metrics["model"] = forecast.model_name
metrics["device"] = str(forecaster.device)

In [22]:
print("--- Results ----------------------------")

for key, value in metrics.items():
    formatted_value = f"{value:.4f}" if isinstance(value, float) else value
    print(f"{key:25}: {formatted_value}")

print()

print("MASE by Variables:")
for column, score in zip(window.columns, per_series_mase):
    print(f"{column:10}: {score:.4f}")

--- Results ----------------------------
mae                      : 1.3032
rmse                     : 1.6230
smape                    : 28.3019
mase                     : 0.9597
inference_time_seconds   : 0.9770
context_length           : 512
prediction_length        : 96
dataset                  : ETTh1
model                    : chronos2
device                   : cuda

MASE by Variables:
HUFL      : 0.8623
HULL      : 1.1896
MUFL      : 0.8566
MULL      : 1.2031
LUFL      : 0.8077
LULL      : 0.6276
OT        : 1.1714


### **Save Predictions**

In [36]:
prediction_dir = project_root / "results" / "forecasts"
metrics_dir = project_root / "results" / "metrics"
prediction_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

In [25]:
rows = []
for variable_index, variable in enumerate(window.columns):
    for step in range(PREDICTION_LENGTH):
        rows.append({
            "dataset": dataset.name,
            "model": forecast.model_name,
            "variable": variable,
            "timestamp": window.target_timestamps[step],
            "horizon_step": step + 1,
            "actual": float(window.target[step, variable_index,]),
            "prediction": float(forecast.predictions[step, variable_index]),
            "lower_10": float(forecast.lower[step, variable_index]),
            "upper_90": float(forecast.upper[step, variable_index])
        })
prediction_df = pd.DataFrame(rows)

In [28]:
prediction_df.head()

,dataset,model,variable,timestamp,horizon_step,actual,prediction,lower_10,upper_90
0,ETTh1,chronos2,HUFL,2018-06-22 20:00:00,1,6.564000,7.734156,6.563862,8.981386
1,ETTh1,chronos2,HUFL,2018-06-22 21:00:00,2,6.564000,8.026466,6.419455,9.356281
2,ETTh1,chronos2,HUFL,2018-06-22 22:00:00,3,9.042000,8.725616,6.995178,10.337739
3,ETTh1,chronos2,HUFL,2018-06-22 23:00:00,4,11.119000,9.912792,7.956967,11.839420
4,ETTh1,chronos2,HUFL,2018-06-23 00:00:00,5,16.476999,10.821527,8.873561,12.811199


In [37]:
prediction_path = prediction_dir / "chronos2_ETTh1_h96.parquet"
prediction_df.to_parquet(prediction_path, index=False)

### **Save Metrics**

In [30]:
metrics["mase_per_variables"] = {
    column: float(score)
    for column, score in zip(window.columns, per_series_mase)}

In [31]:
metrics_path = metrics_dir / "chronos2_ETTh1_h96.json"
with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=4)

In [34]:
print(f"Prediction saved: {prediction_path.relative_to(project_root).as_posix()}")
print(f"Metrics saved: {metrics_path.relative_to(project_root).as_posix()}")

Prediction saved: result/forecasts/chronos2_ETTh1_h96.parquet
Metrics saved: results/metrics/chronos2_ETTh1_h96.json


### **Inspect the Result**

In [41]:
import pandas as pd
df = pd.read_parquet('../../results/forecasts/chronos2_ETTh1_h96.parquet')

In [42]:
df.head()

,dataset,model,variable,timestamp,horizon_step,actual,prediction,lower_10,upper_90
0,ETTh1,chronos2,HUFL,2018-06-22 20:00:00,1,6.564000,7.734156,6.563862,8.981386
1,ETTh1,chronos2,HUFL,2018-06-22 21:00:00,2,6.564000,8.026466,6.419455,9.356281
2,ETTh1,chronos2,HUFL,2018-06-22 22:00:00,3,9.042000,8.725616,6.995178,10.337739
3,ETTh1,chronos2,HUFL,2018-06-22 23:00:00,4,11.119000,9.912792,7.956967,11.839420
4,ETTh1,chronos2,HUFL,2018-06-23 00:00:00,5,16.476999,10.821527,8.873561,12.811199


In [43]:
df.shape

(672, 9)